In [10]:
import numpy as np
import pyscf.gto
import pyscf.scf
import pyscf.cc
import pyscf.fci

from xquces import (
    MolecularHamiltonianLinearOperator,
    hartree_fock_state,
    ucj_from_t_amplitudes,
    ucj_seed_parameters,
    optimize_ucj,
)

bond_length = 2.5
mol = pyscf.gto.M(
    atom=f"H 0 0 0; H 0 0 {bond_length}",
    basis="6-31G",
    unit="Angstrom",
    spin=0,
    charge=0,
    verbose=0,
)

mf = pyscf.scf.RHF(mol)
hf_energy = mf.kernel()

cc = pyscf.cc.CCSD(mf)
_, t1, t2 = cc.kernel()
cc_energy = cc.e_tot

fci_solver = pyscf.fci.FCI(mf)
fci_energy, _ = fci_solver.kernel()

norb = mf.mo_coeff.shape[1]
nelec = (mol.nelectron // 2, mol.nelectron // 2)
nocc = nelec[0]

hf_vec = hartree_fock_state(norb, nelec)
hop = MolecularHamiltonianLinearOperator.from_scf(mf, nelec=nelec)

seed = ucj_from_t_amplitudes(
    t2,
    t1=t1,
    n_layers=1,
)
seed_vec = seed.apply(hf_vec, nelec=nelec)
seed_energy = hop.expectation(seed_vec)

print("HF   =", hf_energy)
print("Seed =", seed_energy)
print("CCSD =", cc_energy)
print("FCI  =", fci_energy)

x0 = ucj_seed_parameters(
    t2,
    t1=t1,
    n_layers=1,
)

ansatz_opt, vec_opt, e_opt, res = optimize_ucj(
    x0,
    hamiltonian=hop,
    reference_vec=hf_vec,
    nocc=nocc,
    n_layers=1,
    method="Powell",
    options={"maxiter": 300, "disp": True},
)

print("Optimized UCJ =", e_opt)
print("success       =", res.success)
print("message       =", res.message)
print("x             =", res.x)

state = ansatz_opt.apply(hf_vec, nelec=nelec)
print("Exact ansatz energy =", hop.expectation(state))

result = run_sqd_from_statevector(
    state,
    norb=norb,
    nelec=nelec,
    hamiltonian=hop,
    n_samples=50000,
    batch_size=64,
    num_batches=8,
    max_iterations=1,
    seed=123,
    initial_occupancies=None,
)

print("SQD energy =", result.energy)
print("determinants in best subspace =", result.best.det_indices)

HF   = -0.8568959429296013
Seed = -0.8568956524068361
CCSD = -1.0008131477741085
FCI  = -1.0008131479968516
Optimization terminated successfully.
         Current function value: -0.859597
         Iterations: 2
         Function evaluations: 327
Optimized UCJ = -0.8595972939143535
success       = True
message       = Optimization terminated successfully.
x             = [ 4.99517370e-03  0.00000000e+00  3.14876209e+00  0.00000000e+00
  1.00000000e+00  5.03040006e-03  7.23210243e-01  4.00160817e-16
  9.37701138e-02  2.97741044e-17 -1.12395672e-09  1.37388542e-01
 -1.60182128e-12  2.66549656e-11 -7.71336499e-05 -5.09739636e-09]
Exact ansatz energy = -0.8595972939143535
SQD energy = -0.8596212043252294
determinants in best subspace = [ 0  8 10]
